# CrewAI: 基于角色的员工和流

CrewAI 是2026年基于角色的多agent 框架。四个原语：Agent、Task、Crew、Process。 两个上层形态：Crews（自动，基于角色的合作）、Flows（事件驱动、决定性）。

## 问题描述

使用多agent框架的团队碰到了同样一堵像。“自动协作”听起来很棒，然后一个用户反馈了一个bug，你需要确定性的流程才能重现。或者财务问LLM路由到的员工每轮花费预算是多少。或者一个电话想知道哪个agent在凌晨3点卡住了。

自由形式基于LLM路由的员工不能干净地回答上述任何问题。纯DAGs可以回答所有问题但是丧失了一个头脑风暴agent所需的探索形态。

CrewAI 的切分对这种取舍是诚实的。Crews 用于写协作、基于角色、探索性任务。 Flows 用于事件驱动、代码控制、可审计的生产。相同的框架、两种形态、按场景调。

## 基本概念

### 四个原语

CrewAI 设计很小，记住这些，剩余的就是配置。

- **Agent**。 角色+目标+背景+工具+（可选）llm。背景是承重的，它塑造语气、判断、以及何时停止。工具是agent可以调用的函数。
- **Task**。 描述+预期输出+agent+（可选）上下文+（可选）输出校验。工作的可复用单元。预期输出是契约，上下文列出哪些上游任务的输出需要传入。输出校验强制结构化形状。
- **Crew**。 容器。管理agents列表、tasks列表、process。可选记忆+详细输出+管理者llm配置。
- **Process**。 执行策略。序列、层级、共识。选择执行的形态。

Agents 不直接关注其他Agents。Tasks 引用 Agents。Crew 是Tasks的序列。Process决定谁挑选下一个Task。这就是模型的心智。

### 序列、层级、共识

- 序列。  任务按照声明的顺序执行。第N个任务的输出是第N+1个任务的输入。成本最低、可预测性最强。当顺序固定的时候使用。
- 层级。  一个经理Agent （独立于LLM调用）在专家间路由。CrewAI 通过你的经理llm配置或者默认部署专家。经理负责在每轮挑选下一个任务，然后选择拒绝或者重新分发。当你有四个或者更多专家，且顺序确实取决于先前输出的时候调用。
- 共识。  计划中、还未实现到API中。是未来机遇投票的流程。

层级在每个专家调用前添加一次LLM调用（经理）。Token消耗增长，只有当你真的需要路由的时候才为它买单。

### Crews 和 Flows

这是2026年文档开篇定义的框架。

- **Crew**。 LLM驱动自动化。这个框架在运行时挑形态。在研究、头脑风暴、首次草案、或者任何路径本身就是一种答案场景下有优势。 但是难重复、难测试。做原型便宜。
- **Flow**。 事件驱动图。`@start`标记入口，`@listen(topic)`标记一个其他步骤发出topic时触发的步骤。在产品化、可观测、可测试、决定性任务中有优势。

2026年文档中的生产建议：从Flow开始，将Crews折叠成`Crew.kickoff()`调用（只在自动化能覆盖开销的时候使用）。Flow给了你审计轨迹、Crew给了你探索。组合，而非挑选。

### 记忆钩子

CrewAI 有四种形式的记忆。是组合态的：一个Crew可以一次性开启四种。
- 短期。 单轮中的会话缓存，最后会擦除。
- 长期。 跨轮次存在。在向量数据库中存储，通过当前任务的相似度做召回。
- 实体。 每个实体的事实。通过实体作为键值做索引，而不是相似度。
- 上下文。  非预加载的、召回时组装的、agent需要的相关记忆。

### CrewAI 适合
- 3到6个具名角色的agent 在一个协作工作流中。提案、审查、计划、头脑风暴等。
- 路由场景，LLM对于下一步的判断本身就是价值的一部分时（如分层）。
- 团队更偏向通过 角色+目标+背景 配置而不是读图定义的场景。

### CrewAI 不适合
- 决定性的、严格排序的DAGs。  使用LangGraph
- 延迟要求高。
- 单agent循环。

### 什么时候模式失效

- **背景导致prompt膨胀**。 如果5个agent，每个agent都有2000+token的背景，在第一次llm调用钱账单就在烧了。建议背景不超过200个词，在agent之间复用措辞，不要把团队风格重复5遍。

- **经理LLM token税**。 层级流程在每次专家调用前都加了一次经理调用。在一个5任务crew上就是6次llm调用，而且经理调用包含完整的任务列表和先前的输出。除非路由取决于输出，否则切换到序列。

- **脆弱的交接**。 第N个任务的期望输出是一份大纲。第N+1个任务将其作为上下文，想解析出3个section，结果llm吐出了4个。下游agent即兴发挥。用Task N 的输出校验解决，输出结构化类型对象，而非自由文本。

- **Crew 当生产**。 自由形式的Crew没有套Flow就投入生产。输出的可变形很大，难以重复，值班的人没办法把一次好的调用和一次坏的调用区分开来。使用一层Flow包裹。

# 开始编码

对应本章核心：**四个原语（Agent / Task / Crew / Process）**、**序列 vs 层级**、**Crew（探索）vs Flow（可审计）**、**输出校验防脆弱交接**。  
先用玩具跑通序列/层级与 Flow 包 Crew；再用 **LangChain + DeepSeek** 挂真实角色协作（不硬凑 PyTorch）。


## 1. 教学玩具：CrewAI 四原语 + Flow

- **Agent / Task / Crew / Process**：Task 引用 Agent；Process 决定下一个 Task。
- **sequential**：按声明顺序；**hierarchical**：经理每轮挑专家任务。
- **output_validator**：强制结构化交接，避免下游即兴发挥。
- **Flow**：`@start` / `@listen`；生产路径里把 `Crew.kickoff()` 折成一步。


In [ ]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

ProcessKind = Literal["sequential", "hierarchical"]
WorkerFn = Callable[[str, dict[str, Any]], str]
ValidatorFn = Callable[[str], Any]


@dataclass
class Agent:
    """角色 + 目标 + 背景 + 执行器（玩具里用函数代替 LLM）。"""

    role: str
    goal: str
    backstory: str
    worker: WorkerFn
    name: str = ""

    def __post_init__(self) -> None:
        if not self.name:
            self.name = self.role

    def run(self, prompt: str, ctx: dict[str, Any]) -> str:
        """
        Args:
            prompt: 任务描述 + 期望输出 + 上游上下文。
            ctx: 短期记忆等。

        Returns:
            text: 原始输出。
        """
        return self.worker(prompt, ctx)


@dataclass
class Task:
    """可复用工作单元：契约在 expected_output；校验防脆弱交接。"""

    description: str
    expected_output: str
    agent: Agent
    context_from: list[str] = field(default_factory=list)
    name: str = ""
    output_validator: ValidatorFn | None = None

    def __post_init__(self) -> None:
        if not self.name:
            self.name = self.description[:24].strip() or self.agent.name


@dataclass
class TaskResult:
    task_name: str
    agent: str
    raw: str
    validated: Any


@dataclass
class CrewResult:
    process: ProcessKind
    results: list[TaskResult]
    audit: list[str] = field(default_factory=list)
    llm_calls: int = 0

    @property
    def output(self) -> Any:
        return self.results[-1].validated if self.results else None


@dataclass
class ShortMemory:
    """单轮会话缓存；kickoff 结束可擦除。"""

    turns: list[str] = field(default_factory=list)

    def add(self, text: str) -> None:
        self.turns.append(text)

    def clear(self) -> None:
        self.turns.clear()


class Crew:
    """Agents + Tasks + Process 容器。"""

    def __init__(
        self,
        agents: list[Agent],
        tasks: list[Task],
        process: ProcessKind = "sequential",
        manager: Agent | None = None,
        memory: ShortMemory | None = None,
    ) -> None:
        self.agents = {a.name: a for a in agents}
        self.tasks = tasks
        self.process = process
        self.manager = manager
        self.memory = memory or ShortMemory()

    def _build_prompt(self, task: Task, done: dict[str, TaskResult]) -> str:
        parts = [
            f"Role: {task.agent.role}",
            f"Goal: {task.agent.goal}",
            f"Backstory: {task.agent.backstory}",
            f"Task: {task.description}",
            f"Expected: {task.expected_output}",
        ]
        if task.context_from:
            ctx_bits = []
            for name in task.context_from:
                if name in done:
                    ctx_bits.append(f"[{name}] {done[name].validated}")
            parts.append("Context:\n" + "\n".join(ctx_bits))
        if self.memory.turns:
            parts.append("ShortMemory: " + " | ".join(self.memory.turns[-3:]))
        return "\n".join(parts)

    def _run_task(self, task: Task, done: dict[str, TaskResult], audit: list[str]) -> TaskResult:
        prompt = self._build_prompt(task, done)
        raw = task.agent.run(prompt, {"done": {k: v.validated for k, v in done.items()}})
        validated: Any = raw
        if task.output_validator is not None:
            validated = task.output_validator(raw)
        self.memory.add(f"{task.name}:{validated}")
        audit.append(f"task:{task.name}@agent:{task.agent.name}")
        return TaskResult(task.name, task.agent.name, raw, validated)

    def kickoff(self) -> CrewResult:
        """
        Returns:
            result: 执行结果 + 审计 + llm_calls（玩具里每次 agent.run 计 1）。
        """
        done: dict[str, TaskResult] = {}
        order: list[TaskResult] = []
        audit: list[str] = []
        calls = 0
        pending = list(self.tasks)

        if self.process == "sequential":
            for task in pending:
                tr = self._run_task(task, done, audit)
                calls += 1
                done[task.name] = tr
                order.append(tr)
        elif self.process == "hierarchical":
            if self.manager is None:
                raise ValueError("hierarchical process requires manager")
            while pending:
                # 经理 LLM token 税：每轮额外 1 次调用
                menu = ", ".join(t.name for t in pending)
                choice = self.manager.run(
                    f"Pick next task from: [{menu}]. Reply with exact task name only.",
                    {"pending": [t.name for t in pending], "done": list(done)},
                ).strip()
                calls += 1
                audit.append(f"manager:pick:{choice}")
                task = next((t for t in pending if t.name == choice), pending[0])
                pending.remove(task)
                tr = self._run_task(task, done, audit)
                calls += 1
                done[task.name] = tr
                order.append(tr)
        else:
            raise ValueError(f"unsupported process: {self.process}")

        return CrewResult(self.process, order, audit, calls)


class Flow:
    """事件驱动、决定性：@start / @listen；可把 Crew.kickoff 折成一步。"""

    def __init__(self) -> None:
        self._start: Callable[[dict[str, Any]], Any] | None = None
        self._listeners: dict[str, list[Callable[[Any, dict[str, Any]], Any]]] = {}
        self.audit: list[str] = []

    def start(self, fn: Callable[[dict[str, Any]], Any]) -> Callable[[dict[str, Any]], Any]:
        """标记入口。"""
        self._start = fn
        return fn

    def listen(self, topic: str) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
        """
        Args:
            topic: 上游 emit 的主题。
        """

        def deco(fn: Callable[..., Any]) -> Callable[..., Any]:
            self._listeners.setdefault(topic, []).append(fn)
            return fn

        return deco

    def emit(self, topic: str, payload: Any, state: dict[str, Any]) -> list[Any]:
        """
        Args:
            topic: 事件名。
            payload: 事件载荷。
            state: 共享流状态（可审计）。

        Returns:
            outs: 各 listener 返回值。
        """
        self.audit.append(f"emit:{topic}")
        outs: list[Any] = []
        for fn in self._listeners.get(topic, []):
            self.audit.append(f"listen:{topic}->{fn.__name__}")
            outs.append(fn(payload, state))
        return outs

    def kickoff(self, inputs: dict[str, Any] | None = None) -> dict[str, Any]:
        """
        Args:
            inputs: 入口参数。

        Returns:
            state: 流结束后的状态（含 audit）。
        """
        if self._start is None:
            raise RuntimeError("Flow has no @start")
        self.audit = []
        state: dict[str, Any] = dict(inputs or {})
        state["audit"] = self.audit
        self.audit.append("start")
        first = self._start(state)
        state["start_result"] = first
        # 约定：start 返回 (topic, payload) 则继续派发
        if isinstance(first, tuple) and len(first) == 2 and isinstance(first[0], str):
            topic, payload = first
            state["last"] = self.emit(topic, payload, state)
        return state


def validate_outline_3(text: str) -> dict[str, Any]:
    """
    期望恰好 3 个 section（脆弱交接的修法）。

    Args:
        text: 原始大纲文本。

    Returns:
        obj: {"sections": [...]} 长度必须为 3。
    """
    sections = [s.strip() for s in re.split(r"[;\n]+", text) if s.strip()]
    if len(sections) != 3:
        raise ValueError(f"expected 3 sections, got {len(sections)}: {sections}")
    return {"sections": sections}


print("CrewAI toy ready | Agent Task Crew Process Flow")


## 2. 玩具示例：序列 / 层级 / 校验 / Flow 包 Crew


In [ ]:
def demo_crewai_toy() -> None:
    """断言四原语、两种 Process、输出校验、Flow 审计。"""

    def researcher(prompt: str, ctx: dict[str, Any]) -> str:
        return "factA; factB; factC"

    def writer(prompt: str, ctx: dict[str, Any]) -> str:
        sections = ctx["done"].get("outline", {}).get("sections") or ["a", "b", "c"]
        return " / ".join(f"para({s})" for s in sections)

    def reviewer(prompt: str, ctx: dict[str, Any]) -> str:
        draft = ctx["done"].get("draft", "")
        return f"APPROVED:{draft}"

    def bad_outline(prompt: str, ctx: dict[str, Any]) -> str:
        return "one; two; three; four"  # 4 sections → 校验应失败

    def manager_pick(prompt: str, ctx: dict[str, Any]) -> str:
        pending = ctx.get("pending") or []
        # 故意非声明顺序：先 reviewer 任务名不在首；按 research→outline→draft
        for prefer in ("research", "outline", "draft", "review"):
            if prefer in pending:
                return prefer
        return pending[0]

    ag_r = Agent("Researcher", "find facts", "concise notes", researcher, name="researcher")
    ag_o = Agent("Outliner", "structure", "exactly 3 sections", researcher, name="outliner")
    # outliner 用固定三节
    ag_o.worker = lambda p, c: "Intro; Body; Outro"
    ag_w = Agent("Writer", "draft", "tight prose", writer, name="writer")
    ag_v = Agent("Reviewer", "approve", "strict", reviewer, name="reviewer")
    ag_m = Agent("Manager", "route", "pick next", manager_pick, name="manager")

    t_research = Task("gather facts", "bullet facts", ag_r, name="research")
    t_outline = Task(
        "make outline",
        "3 sections",
        ag_o,
        context_from=["research"],
        name="outline",
        output_validator=validate_outline_3,
    )
    t_draft = Task("write draft", "prose", ag_w, context_from=["outline"], name="draft")
    t_review = Task("review", "approve/reject", ag_v, context_from=["draft"], name="review")

    # 1) 序列：可预测，calls == len(tasks)
    seq = Crew([ag_r, ag_o, ag_w, ag_v], [t_research, t_outline, t_draft, t_review], process="sequential")
    r_seq = seq.kickoff()
    assert r_seq.process == "sequential"
    assert r_seq.llm_calls == 4
    assert str(r_seq.output).startswith("APPROVED:")
    print("sequential:", r_seq.audit, "calls=", r_seq.llm_calls)

    # 2) 层级：经理每轮 +1 → calls == 2 * n_tasks
    hier_tasks = [
        Task("gather facts", "bullets", ag_r, name="research"),
        Task(
            "make outline",
            "3 sections",
            ag_o,
            context_from=["research"],
            name="outline",
            output_validator=validate_outline_3,
        ),
        Task("write draft", "prose", ag_w, context_from=["outline"], name="draft"),
    ]
    hier = Crew([ag_r, ag_o, ag_w], hier_tasks, process="hierarchical", manager=ag_m)
    r_hier = hier.kickoff()
    assert r_hier.llm_calls == 6  # 3 manager + 3 expert
    assert any(a.startswith("manager:pick:") for a in r_hier.audit)
    print("hierarchical calls=", r_hier.llm_calls, r_hier.audit)

    # 3) 脆弱交接：无校验会静默传 4 节；有校验则抛错
    ag_bad = Agent("BadOutliner", "x", "x", bad_outline, name="bad")
    t_bad = Task("outline", "3 sections", ag_bad, name="outline", output_validator=validate_outline_3)
    try:
        Crew([ag_bad], [t_bad]).kickoff()
        raise AssertionError("expected validation error")
    except ValueError as e:
        assert "expected 3 sections" in str(e)
        print("validation blocked fragile handoff:", e)

    # 4) Flow 包 Crew：可审计轨迹
    flow = Flow()
    crew_for_flow = Crew(
        [ag_r, ag_o, ag_w],
        [
            Task("gather facts", "bullets", ag_r, name="research"),
            Task(
                "make outline",
                "3 sections",
                ag_o,
                context_from=["research"],
                name="outline",
                output_validator=validate_outline_3,
            ),
            Task("write draft", "prose", ag_w, context_from=["outline"], name="draft"),
        ],
        process="sequential",
    )

    @flow.start
    def begin(state: dict[str, Any]) -> tuple[str, Any]:
        topic = state.get("topic", "blog")
        return "run_crew", {"topic": topic}

    @flow.listen("run_crew")
    def run_crew_step(payload: Any, state: dict[str, Any]) -> Any:
        result = crew_for_flow.kickoff()
        state["crew_output"] = result.output
        state["crew_calls"] = result.llm_calls
        flow.emit("notify", {"ok": True, "output": result.output}, state)
        return result.output

    @flow.listen("notify")
    def notify(payload: Any, state: dict[str, Any]) -> str:
        state["notified"] = True
        return "notified"

    st = flow.kickoff({"topic": "CrewAI"})
    assert st.get("notified") is True
    assert "start" in st["audit"] and any(x.startswith("emit:run_crew") for x in st["audit"])
    print("flow audit:", st["audit"])
    print("TOY DEMO OK")


demo_crewai_toy()


## 3. 生产级：Crew + Flow + LangChain / DeepSeek

真实 LLM 充当 Agent.worker；工具暴露 `kickoff_crew`（序列探索）与 `kickoff_flow`（Flow 包 Crew，带审计）。  
大纲 Task 带 JSON 校验（恰好 3 个 sections）。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
LAST_CREW: CrewResult | None = None
LAST_FLOW: dict[str, Any] | None = None


def get_llm(*, temperature: float = 0.2) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def llm_worker(system: str) -> WorkerFn:
    """
    Args:
        system: 角色系统提示（保持短背景，避免 prompt 膨胀）。

    Returns:
        worker: Agent.worker。
    """

    def _run(prompt: str, ctx: dict[str, Any]) -> str:
        full = f"{system}\n\n{prompt}\n\nReply in Chinese. Be concise."
        return str(get_llm().invoke(full).content).strip()

    return _run


def validate_sections_json(text: str) -> dict[str, Any]:
    """
    解析 LLM 输出为恰好 3 个 section。

    Args:
        text: 模型原文。

    Returns:
        obj: {"sections": [s1,s2,s3]}。
    """
    m = re.search(r"\{[\s\S]*\}", text)
    raw = m.group(0) if m else text
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        # 退回分号切分
        return validate_outline_3(text)
    sections = data.get("sections") if isinstance(data, dict) else None
    if not isinstance(sections, list) or len(sections) != 3:
        raise ValueError(f"expected JSON sections[3], got {data!r}")
    return {"sections": [str(s).strip() for s in sections]}


def build_content_crew(topic: str, process: ProcessKind = "sequential") -> Crew:
    """
    Args:
        topic: 主题。
        process: sequential / hierarchical。

    Returns:
        crew: 研究→大纲(校验)→起草。
    """
    researcher = Agent(
        role="研究员",
        goal="给出与主题相关的要点",
        backstory="只写事实要点，不写散文。",
        worker=llm_worker("You are a researcher. Output 3 short facts separated by semicolons."),
        name="researcher",
    )
    outliner = Agent(
        role="大纲编辑",
        goal="产出恰好 3 个章节标题",
        backstory="严格 JSON，禁止额外章节。",
        worker=llm_worker(
            'You are an outliner. Reply ONLY JSON: {"sections":["...","...","..."]} with exactly 3 items.'
        ),
        name="outliner",
    )
    writer = Agent(
        role="撰稿人",
        goal="按三节写短文",
        backstory="中文、总长不超过 120 字。",
        worker=llm_worker("You are a writer. Use the 3 sections from context. One short paragraph each."),
        name="writer",
    )
    manager = Agent(
        role="经理",
        goal="选择下一个任务",
        backstory="只回复任务名。",
        worker=llm_worker(
            "You are a crew manager. Reply with exactly one pending task name: research|outline|draft."
        ),
        name="manager",
    )

    tasks = [
        Task(
            description=f"Research topic: {topic}",
            expected_output="3 facts separated by semicolons",
            agent=researcher,
            name="research",
        ),
        Task(
            description=f"Outline for: {topic}",
            expected_output='JSON {"sections":[a,b,c]}',
            agent=outliner,
            context_from=["research"],
            name="outline",
            output_validator=validate_sections_json,
        ),
        Task(
            description=f"Draft article on: {topic}",
            expected_output="short Chinese draft",
            agent=writer,
            context_from=["outline", "research"],
            name="draft",
        ),
    ]
    return Crew(
        [researcher, outliner, writer],
        tasks,
        process=process,
        manager=manager if process == "hierarchical" else None,
    )


def build_prod_flow(topic: str) -> Flow:
    """Flow 入口 → listen(run_crew) → listen(notify)。"""
    flow = Flow()
    crew = build_content_crew(topic, process="sequential")

    @flow.start
    def begin(state: dict[str, Any]) -> tuple[str, Any]:
        return "run_crew", {"topic": state.get("topic", topic)}

    @flow.listen("run_crew")
    def run_crew_step(payload: Any, state: dict[str, Any]) -> Any:
        result = crew.kickoff()
        state["crew_output"] = result.output
        state["crew_audit"] = result.audit
        state["crew_calls"] = result.llm_calls
        flow.emit("notify", {"output": result.output}, state)
        return result.output

    @flow.listen("notify")
    def notify(payload: Any, state: dict[str, Any]) -> str:
        state["notified"] = True
        return "ok"

    return flow


class KickoffCrewArgs(BaseModel):
    topic: str = Field(description="Content topic")
    process: ProcessKind = Field(default="sequential", description="sequential or hierarchical")


class KickoffFlowArgs(BaseModel):
    topic: str = Field(description="Content topic for audited Flow wrapping Crew")


class InspectArgs(BaseModel):
    pass


def kickoff_crew_impl(topic: str, process: ProcessKind = "sequential") -> str:
    """
    Returns:
        json: Crew 结果摘要。
    """
    global LAST_CREW
    crew = build_content_crew(topic, process=process)
    # hierarchical manager 可能不回精确任务名：做一次安全回退
    try:
        result = crew.kickoff()
    except Exception as e:
        return json.dumps({"error": str(e)}, ensure_ascii=False)
    LAST_CREW = result
    return json.dumps(
        {
            "process": result.process,
            "llm_calls": result.llm_calls,
            "audit": result.audit,
            "output": result.output,
            "steps": [{"task": t.task_name, "agent": t.agent, "validated": t.validated} for t in result.results],
        },
        ensure_ascii=False,
        default=str,
    )


def kickoff_flow_impl(topic: str) -> str:
    """
    Returns:
        json: Flow 状态（含 audit）。
    """
    global LAST_FLOW
    flow = build_prod_flow(topic)
    state = flow.kickoff({"topic": topic})
    LAST_FLOW = state
    return json.dumps(
        {
            "audit": state.get("audit"),
            "crew_calls": state.get("crew_calls"),
            "crew_audit": state.get("crew_audit"),
            "crew_output": state.get("crew_output"),
            "notified": state.get("notified"),
        },
        ensure_ascii=False,
        default=str,
    )


def inspect_last_impl() -> str:
    """
    Returns:
        json: 最近一次 Crew / Flow 快照。
    """
    return json.dumps(
        {
            "last_crew_calls": None if LAST_CREW is None else LAST_CREW.llm_calls,
            "last_crew_audit": None if LAST_CREW is None else LAST_CREW.audit,
            "last_flow_audit": None if LAST_FLOW is None else LAST_FLOW.get("audit"),
        },
        ensure_ascii=False,
    )


def build_crewai_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: Crew / Flow 三件套。
    """

    def _crew(**kwargs: Any) -> str:
        a = KickoffCrewArgs(**kwargs)
        return kickoff_crew_impl(a.topic, a.process)

    def _flow(**kwargs: Any) -> str:
        return kickoff_flow_impl(KickoffFlowArgs(**kwargs).topic)

    def _inspect(**kwargs: Any) -> str:
        return inspect_last_impl()

    return [
        StructuredTool.from_function(
            name="kickoff_crew",
            description="Run a role-based Crew (sequential cheap; hierarchical adds manager token tax).",
            func=_crew,
            args_schema=KickoffCrewArgs,
        ),
        StructuredTool.from_function(
            name="kickoff_flow",
            description="Production path: deterministic Flow that wraps Crew.kickoff with audit trail.",
            func=_flow,
            args_schema=KickoffFlowArgs,
        ),
        StructuredTool.from_function(
            name="inspect_last",
            description="Inspect last Crew/Flow audit and call counts.",
            func=_inspect,
            args_schema=InspectArgs,
        ),
    ]


CREWAI_TOOLS = build_crewai_tools()


def build_control_agent() -> Any:
    """
    Returns:
        agent: 通过工具驱动 Crew / Flow。
    """
    system = (
        "You operate a CrewAI-style system via tools.\n"
        "Prefer kickoff_flow for production-like audited runs; use kickoff_crew for exploration.\n"
        "After a run, call inspect_last. Summarize in Chinese."
    )
    return create_agent(get_llm(), CREWAI_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 600 else str(m.content)[:600] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


print(f"CrewAI-style + LangChain ready | {MODEL}")


## 4. 生产示例：序列 Crew + Flow 包 Crew


In [ ]:
def demo_deepseek_crewai() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    print("=== sequential crew ===")
    crew_json = kickoff_crew_impl("角色模型与消息传递", process="sequential")
    crew_data = json.loads(crew_json)
    print(json.dumps(crew_data, ensure_ascii=False, indent=2)[:900])
    assert "error" not in crew_data
    assert crew_data.get("llm_calls") == 3
    assert isinstance(crew_data.get("output"), str) and crew_data["output"]

    print("=== flow wraps crew ===")
    flow_json = kickoff_flow_impl("Crew 与 Flow 如何组合")
    flow_data = json.loads(flow_json)
    print(json.dumps(flow_data, ensure_ascii=False, indent=2)[:900])
    assert flow_data.get("notified") is True
    assert flow_data.get("crew_output")
    assert any(str(x).startswith("emit:") for x in (flow_data.get("audit") or []))

    print("=== control agent ===")
    try:
        agent = build_control_agent()
        result = agent.invoke(
            {
                "messages": [
                    HumanMessage(
                        content=(
                            "用 kickoff_flow 跑主题「为何生产要用 Flow 包 Crew」，"
                            "再 inspect_last，用中文总结审计轨迹与产出。"
                        )
                    )
                ]
            }
        )
        print(format_agent_messages(result["messages"]))
        assert count_tool_calls(result["messages"]) >= 2
    except Exception as e:
        print(f"control agent skipped due to LLM error: {type(e).__name__}: {e}")
    print("PRODUCTION DEMO OK")


demo_deepseek_crewai()
